In [ ]:
%reset -f
import arcpy
arcpy.ResetEnvironments()
import os

In [ ]:
# ── PATHS ────────────────────────────────────────────────────────────────────────────────────────────

# Input LF and HF interpolated backgroud/mission path data CSVs
lf_csv = r"C:\Users\kkeii\Documents\THESIS_REPOSITORIES\MISSION_REPOSITORIES\UVI2_Around_STX_20250306\data\4_data_interpolated\LF_background_INTERPOLATED_UVI2_Around_STX_20250306.csv"
hf_csv = r"C:\Users\kkeii\Documents\THESIS_REPOSITORIES\MISSION_REPOSITORIES\UVI2_Around_STX_20250306\data\4_data_interpolated\HF_background_INTERPOLATED_UVI2_Around_STX_20250306.csv"


# intermediate point feature classes output by 'XY Table to Point' tool before reprojection (*output to misc_around_stx_2025.gdb)
lf_deploy_g4326 = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\misc_around_stx_2025.gdb\LF_deploy_pt_g4326"
hf_deploy_g4326 = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\misc_around_stx_2025.gdb\HF_deploy_pt_g4326"
lf_recover_g4326 = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\misc_around_stx_2025.gdb\LF_recover_pt_g4326"
hf_recover_g4326 = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\misc_around_stx_2025.gdb\HF_recover_pt_g4326"


# Output DEPLOYMENT LOCATION points - glider icon (reprojected to EPSG:26920, *output to GIS_UVI2_Around_STX_20250306.gdb)
lf_deploy_p26920 = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\GIS_UVI2_Around_STX_20250306.gdb\LF_deploy_pt_p26920"
hf_deploy_p26920 = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\GIS_UVI2_Around_STX_20250306.gdb\HF_deploy_pt_p26920"

# Output RECOVERY LOCATION points - flag icon (reprojected to EPSG:26920, *output to GIS_UVI2_Around_STX_20250306.gdb)
lf_recover_p26920 = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\GIS_UVI2_Around_STX_20250306.gdb\LF_recover_pt_p26920"
hf_recover_p26920 = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\GIS_UVI2_Around_STX_20250306.gdb\HF_recover_pt_p26920"

In [ ]:
# ── SETTINGS ─────────────────────────────────────────────────────────────────────────────────────────
arcpy.env.overwriteOutput = True    # overwrite existing outputs on rerun

In [ ]:
# ── 1. CREATE MISSION START/END POINT FEATURE CLASSES FROM CSV DATA ──────────────────────────────────

# XY Table To Point (Data Management Tools): Function
def xy_table_to_point(in_csv, out_points_g4326, x_field, y_field):

    try:
        arcpy.management.XYTableToPoint(
            in_table=in_csv,
            out_feature_class=out_points_g4326,
            x_field=x_field,
            y_field=y_field,
            z_field=None,
            # output point feature class in EPSG:4326 (native coordinate system of glider data that background coordinates are interpolated from)
            coordinate_system = arcpy.SpatialReference(4326)
        )
        print(f"✅ Created point feature class: {out_points_g4326} \n")

    # print warning message if tool fails
    except Exception as e:
        print(f"❌[FAILURE] Point feature class creation failed for {os.path.basename(out_points_g4326)}: {e} \n")



# Call function for LF and HF deployment points (* use start coordinates)
xy_table_to_point(lf_csv, lf_deploy_g4326, "start_longitude_DD", "start_latitude_DD")
xy_table_to_point(hf_csv, hf_deploy_g4326, "start_longitude_DD", "start_latitude_DD")

# Call function for LF and HF recovery points (* use end coordinates)
xy_table_to_point(lf_csv, lf_recover_g4326, "end_longitude_DD", "end_latitude_DD")
xy_table_to_point(hf_csv, hf_recover_g4326, "end_longitude_DD", "end_latitude_DD")

In [ ]:
# ── 2. REPROJECT FEATURE CLASSES CREATED FROM CSV DATA TO EPSG:26920 ─────────────────────────────────

# Project (Data Management Tools): Function
def reproject_fc(in_fc, out_fc):
    
    try:
        arcpy.management.Project(
            in_dataset=in_fc,
            out_dataset=out_fc,
            # output coordinate system
            out_coor_system=arcpy.SpatialReference(26920),
            transform_method="WGS_1984_(ITRF00)_To_NAD_1983",
            # input coordinate system
            in_coor_system=arcpy.SpatialReference(4326),
            # * do NOT want tool to add extra vertices to feature classes (want to preserve input data)
            preserve_shape="NO_PRESERVE_SHAPE"
        )
        print(f"✅ Reprojected feature class: {out_fc} \n")

    # print warning message if tool fails
    except Exception as e:
        print(f"❌[FAILURE] Reprojection failed for {os.path.basename(out_fc)}: {e} \n")


# Call function for LF and HF mission deploy/recovery point feature classes
reproject_fc(lf_deploy_g4326, lf_deploy_p26920)
reproject_fc(hf_deploy_g4326, hf_deploy_p26920)
reproject_fc(lf_recover_g4326,lf_recover_p26920)
reproject_fc(hf_recover_g4326, hf_recover_p26920)

In [ ]:
# ── 3. DEPLOYMENT POINTS - REMOVE ALL ROWS IN ATTRIBUTE TABLE *EXCEPT* STARTING POINT ────────────────

# Function to find row in Attribute Table with earliest datetime, delete all other rows, and print messages with earliest datetime found + number rows in Attribute Table after removal
def keep_earliest_point(in_fc):

    # print header identifying which feature class this block reports on
    print(f"\n------------- {os.path.basename(in_fc)} -------------\n")

    # open read-only cursor on datetime field
    with arcpy.da.SearchCursor(in_fc, ["start_datetime_UTC"]) as cur:
        # scan all rows and grab minimum (earliest) datetime
        earliest = min(row[0] for row in cur)

    # print earliest datetime found
    print(f"  ⏱️ Earliest start_datetime_UTC: {earliest}")

    # open editable cursor on datetime field
    with arcpy.da.UpdateCursor(in_fc, ["start_datetime_UTC"]) as cur:
        # track whether earliest row has already been kept
        kept = False
        # step through every row
        for row in cur:
            # keep first row matching the earliest datetime
            if row[0] == earliest and not kept:
                # flag row so no further matches are kept
                kept = True
            else:
                # delete every other row
                cur.deleteRow()

    # report how many rows are left (should be 1)
    print(f"  📑 Rows remaining: {arcpy.management.GetCount(in_fc)[0]}")


# Call function for HF and LF deployment point feature classes
keep_earliest_point(lf_deploy_p26920)
keep_earliest_point(hf_deploy_p26920)

In [ ]:
# ── 4. RECOVERY POINTS: REMOVE ALL ROWS IN ATTRIBUTE TABLE *EXCEPT* ENDING POINT ─────────────────────

# Function to find row in Attribute Table with latest datetime, delete all other rows, and print messages with latest datetime found + number rows in Attribute Table after removal
def keep_latest_point(in_fc):

    # print header identifying which feature class this block reports on
    print(f"\n------------- {os.path.basename(in_fc)} -------------\n")

    # open read-only cursor on datetime field
    with arcpy.da.SearchCursor(in_fc, ["end_datetime_UTC"]) as cur:
        # scan all rows and grab maximum (latest) datetime
        latest = max(row[0] for row in cur)

    # print latest datetime found
    print(f"  ⏱️ Latest end_datetime_UTC: {latest}")

    # open editable cursor on datetime field
    with arcpy.da.UpdateCursor(in_fc, ["end_datetime_UTC"]) as cur:
        # track whether latest row has already been kept
        kept = False
        # step through every row
        for row in cur:
            # keep first row matching the latest datetime
            if row[0] == latest and not kept:
                # flag row so no further matches are kept
                kept = True
            else:
                # delete every other row
                cur.deleteRow()

    # report how many rows are left (should be 1)
    print(f"  📑 Rows remaining: {arcpy.management.GetCount(in_fc)[0]}")


# Call function for HF and LF recovery point feature classes
keep_latest_point(lf_recover_p26920)
keep_latest_point(hf_recover_p26920)

In [ ]:
# ── 5. ADD AND POPULATE 'LABEL' FIELD IN ATTRIBUTE TABLES ────────────────────────────────────────────

# Function to add 'label' text field to point feature class Attribute Tables and populate it with a given string
def add_label(in_fc, label_text):

    try:
        # Add Field (Data Management Tools):
        arcpy.management.AddField(
            in_table=in_fc,
            field_name="label",
            field_type="TEXT"
            # default field length = 255 characters
        )

        # populate 'label' column with string "Deployment Location" or "Recovery Location" based on input feature class
        with arcpy.da.UpdateCursor(in_fc, ["label"]) as cursor:
            # loop through each row in point fc Attribute Table
            for row in cursor:
                row[0] = label_text
                # write value back to row
                cursor.updateRow(row)

        print(f"✅ 'Label' field added and populated in {os.path.basename(in_fc)}: '{label_text}'")

    # print warning message if tool/population fails
    except Exception as e:
        print(f"❌[FAILURE] 'label' field creation/population failed for {os.path.basename(in_fc)}: {e} \n")


# Call function for deployment point feature classes using string "Deployment Location"
add_label(lf_deploy_p26920, "Deployment Location")
add_label(hf_deploy_p26920, "Deployment Location")

# Call function for recovery point feature classes using string "Recovery Location"
add_label(lf_recover_p26920, "Recovery Location")
add_label(hf_recover_p26920, "Recovery Location")